In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
import pandas as pd

# Relative "../data/..." only works if Jupyter's cwd happens to be docs/;
# if it's launched from the repo root instead, this breaks. Check both.
def _find_repo_root():
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "data" / "set A corporate_rating.csv").exists():
            return candidate
    return Path.cwd()

DATA_PATH = _find_repo_root() / "data" / "set A corporate_rating.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Resolved DATA_PATH:", DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
def group_rating(rating):
    if rating in ["AAA", "AA", "A"]:
        return "Investment-High"
    elif rating == "BBB":
        return "Investment-Low"
    elif rating in ["BB", "B"]:
        return "Speculative"
    elif rating in ["CCC", "CC", "C", "D"]:
        return "Distressed"
    else:
        return "Unknown"

df["Risk_Category"] = df["Rating"].apply(group_rating)

df["Risk_Category"].value_counts()

In [ ]:
# Figure 4.2.2.1

# Define the financial metric groups and their corresponding metrics

metric_groups = {
    "Liquidity": ["currentRatio", "quickRatio", "cashRatio"],
    "Profitability": [
        "netProfitMargin", "pretaxProfitMargin", "grossProfitMargin",
        "operatingProfitMargin", "returnOnAssets", "returnOnCapitalEmployed",
        "returnOnEquity"
    ],
    "Efficiency": ["assetTurnover", "fixedAssetTurnover", "payablesTurnover"],
    "Leverage": ["debtEquityRatio", "debtRatio", "companyEquityMultiplier"],
    "Cash Flow": [
        "freeCashFlowOperatingCashFlowRatio", "freeCashFlowPerShare",
        "cashPerShare", "operatingCashFlowPerShare",
        "operatingCashFlowSalesRatio"
    ],
    "Valuation / Other": [
        "effectiveTaxRate", "ebitPerRevenue", "enterpriseValueMultiple",
        "daysOfSalesOutstanding"
    ]
}

# Count the number of metrics in each group
group_counts = {group: len(cols) for group, cols in metric_groups.items()}

# Plot the number of financial metrics by category
plt.figure(figsize=(9, 5))
plt.bar(group_counts.keys(), group_counts.values())
plt.title("Number of Financial Metrics by Category")
plt.xlabel("Financial Metric Category")
plt.ylabel("Number of Metrics")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 4.2.2.2

# Average financial ratios by credit risk category using standardised values

available_metrics = [
    col
    for cols in metric_groups.values()
    for col in cols
    if col in df.columns
]

# Calculate average value of each financial ratio by risk category
risk_metric_mean = df.groupby("Risk_Category")[available_metrics].mean()

# Standardise each financial ratio so different scales can be compared fairly
plot_data = risk_metric_mean.copy()

plot_data = (plot_data - plot_data.mean()) / plot_data.std()

# Transpose for horizontal bar chart
plot_data = plot_data.T

# Clean feature names for graph labels
plot_data.index = (
    plot_data.index
    .str.replace(r"([a-z])([A-Z])", r"\1 \2", regex=True)
    .str.replace(" / ", "/")
)

# Plot graph
ax = plot_data.plot(
    kind="barh",
    figsize=(12, max(8, 0.45 * len(plot_data))),
    width=0.8
)

ax.set_title("Standardised Average Financial Ratios by Credit Risk Category", pad=12)
ax.set_xlabel("Standardised Average Value")
ax.set_ylabel("Financial Ratio")
ax.tick_params(axis="y", labelsize=9)
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.legend(title="Risk Category", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# Figure 4.2.3.1

# Boxplot of selected financial ratios for outlier detection
selected_metrics = [
    "currentRatio",
    "quickRatio",
    "cashRatio",
    "netProfitMargin",
    "returnOnAssets",
    "returnOnEquity",
    "debtRatio",
    "debtEquityRatio",
    "assetTurnover",
    "operatingCashFlowSalesRatio"
]

available_metrics = [col for col in selected_metrics if col in df.columns]

print("Metrics used for boxplot:")
print(available_metrics)

plt.figure(figsize=(12, 6))

df[available_metrics].boxplot()

plt.yscale("symlog")

plt.title("Boxplot of Selected Financial Ratios for Outlier Detection")
plt.xlabel("Financial Ratios")
plt.ylabel("Value (Symmetric Log Scale)")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# Data quality assessment: missing values and duplicates

print("Missing values per column:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nNumber of duplicated rows:", df.duplicated().sum())

In [ ]:
# Table 4.2.3.2
# IQR-Based Outlier Count for Selected Financial Ratios


selected_metrics = [
    "currentRatio",
    "quickRatio",
    "cashRatio",
    "netProfitMargin",
    "returnOnAssets",
    "returnOnEquity",
    "debtRatio",
    "debtEquityRatio",
    "assetTurnover",
    "operatingCashFlowSalesRatio"
]

available_metrics = [col for col in selected_metrics if col in df.columns]

outlier_summary = []

for col in available_metrics:
    # Convert column to numeric to avoid errors
    numeric_col = pd.to_numeric(df[col], errors="coerce")
    
    Q1 = numeric_col.quantile(0.25)
    Q3 = numeric_col.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outlier_count = ((numeric_col < lower_bound) | (numeric_col > upper_bound)).sum()
    
    outlier_summary.append({
        "Financial Ratio": col,
        "Lower Bound": round(lower_bound, 4),
        "Upper Bound": round(upper_bound, 4),
        "Outlier Count": int(outlier_count)
    })

outlier_summary_df = pd.DataFrame(outlier_summary)
display(outlier_summary_df)